In [ ]:
import os
os.environ["HF_TOKEN"]="XXXXXXX"

### Installing Libraries

In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-huggingface faiss-cpu tiktoken python-dotenv

In [3]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


### Step 1.A : Indexing (Document Ingestion)

In [4]:
# from youtube_transcript_api import YouTubeTranscriptApi
# from youtube_transcript_api._errors import TranscriptsDisabled

video_id = "SwQhKFMxmDY" #only the video id, not the entire url
try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched_transcript.to_raw_data()  #
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")



Hey everybody, welcome to the podcast. My guest today is Dr. Andrew Huberman. Andrew is a neuroscientist. He's a Neurobiology Professor at Stanford Medical School and McKnight Foundation and Pew Foundation Fellow and the founder of Huberman Lab, where he's involved in all kinds of really amazing breakthrough research on brain function, brain plasticity and brain regeneration. His work has been published in top journals like Nature. He's been featured in publications like Time, Scientific America and the BBC. And he's here today to discuss the brain, to discuss growth mindset, how to focus, how to navigate the stressful times and many other subjects. It's an incredible conversation. I think you guys are gonna enjoy it. I appreciate you watching make sure to hit that subscribe button and without further ado, this is me and Dr. Andrew Huberman. First of all, thanks for doing this. I appreciate you coming out. Yeah, my pleasure.
Long time coming. I'm glad we're doing it in person- Likewise

In [5]:
print(fetched_transcript)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Hey everybody, welcome to the podcast.', start=0.4, duration=2.47), FetchedTranscriptSnippet(text='My guest today is Dr. Andrew Huberman.', start=2.87, duration=2.62), FetchedTranscriptSnippet(text='Andrew is a neuroscientist.', start=5.49, duration=2.5), FetchedTranscriptSnippet(text="He's a Neurobiology Professor at Stanford Medical School", start=7.99, duration=3.88), FetchedTranscriptSnippet(text='and McKnight Foundation and Pew Foundation Fellow', start=11.87, duration=3.44), FetchedTranscriptSnippet(text="and the founder of Huberman Lab, where he's involved", start=15.31, duration=2.37), FetchedTranscriptSnippet(text='in all kinds of really amazing breakthrough research', start=17.68, duration=2.26), FetchedTranscriptSnippet(text='on brain function, brain plasticity and brain regeneration.', start=19.94, duration=4.32), FetchedTranscriptSnippet(text='His work has been published in top journals like Nature.', start=24.26, 

In [6]:
print(transcript_list)

[{'text': 'Hey everybody, welcome to the podcast.', 'start': 0.4, 'duration': 2.47}, {'text': 'My guest today is Dr. Andrew Huberman.', 'start': 2.87, 'duration': 2.62}, {'text': 'Andrew is a neuroscientist.', 'start': 5.49, 'duration': 2.5}, {'text': "He's a Neurobiology Professor at Stanford Medical School", 'start': 7.99, 'duration': 3.88}, {'text': 'and McKnight Foundation and Pew Foundation Fellow', 'start': 11.87, 'duration': 3.44}, {'text': "and the founder of Huberman Lab, where he's involved", 'start': 15.31, 'duration': 2.37}, {'text': 'in all kinds of really amazing breakthrough research', 'start': 17.68, 'duration': 2.26}, {'text': 'on brain function, brain plasticity and brain regeneration.', 'start': 19.94, 'duration': 4.32}, {'text': 'His work has been published in top journals like Nature.', 'start': 24.26, 'duration': 3.46}, {'text': "He's been featured in publications", 'start': 27.72, 'duration': 1.34}, {'text': 'like Time, Scientific America and the BBC.', 'start': 

### Step 1.B : Indexing (Text Splitting)

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.create_documents([transcript])

In [8]:
len(chunks)

359

In [9]:
chunks[0]

Document(metadata={}, page_content="Hey everybody, welcome to the podcast. My guest today is Dr. Andrew Huberman. Andrew is a neuroscientist. He's a Neurobiology Professor at Stanford Medical School and McKnight Foundation and Pew Foundation Fellow and the founder of Huberman Lab, where he's involved in all kinds of really amazing breakthrough research on brain function, brain plasticity and brain regeneration. His work has been published in top journals like Nature. He's been featured in publications like Time, Scientific America")

### Step 1.C & 1.D : Indexing (Embedding Creation & Vector Store)

In [10]:
!pip install --upgrade pillow sentence-transformers transformers faiss-cpu

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
vectorstore.index_to_docstore_id

{0: 'e653f109-560c-4808-93c7-770b52ac3a1a',
 1: '52add79e-4315-432b-80c5-6ddcb6e2ebf0',
 2: '725ffe9a-613c-4ebd-ab35-2dcfcc211247',
 3: 'b9a530c9-35df-45cd-9fe0-40f0008c7c61',
 4: 'e20190be-d4cc-41ef-a0b2-aa9787e697ca',
 5: 'a0fcced6-b6d0-40db-bcac-ec9f0c106e5e',
 6: '326fd35f-c9cc-4d2d-9752-9ac0ebb6f0e2',
 7: '3b42ec85-6bcf-4167-8e8f-21b531dcd1f5',
 8: 'db3500b3-9e86-4726-a21f-c2c6ad4380b3',
 9: '522529df-30c1-4b64-8763-3c0d5882fab1',
 10: 'a4144d16-2d85-4b77-bc67-c88d515d1fc2',
 11: '440b27f7-9322-4373-ad4b-f06bc286e473',
 12: '5ecac25e-87c5-485b-b865-fb675d255c99',
 13: 'f0c2e6b5-d13a-4695-8d7e-997318d172e1',
 14: '1d03c125-4686-4c24-b49c-3c3b9aabcc05',
 15: '5f626a66-5ba9-48c1-a4a9-089e4e3c98ee',
 16: '00837c5d-d252-471e-9d7e-c994b1e4af08',
 17: 'd5a89f03-f3ed-4eda-b707-0ba1ec14e8cc',
 18: 'e7b8a99d-1ce1-412b-a118-f47da83db94d',
 19: '4a2e7c0c-8423-4c01-bd72-f7f9112abace',
 20: 'ba9cb8ef-7a73-41c7-8e49-02a2aa9b2052',
 21: 'ddcd77a0-b33e-4035-8c9a-5d111db0d235',
 22: '2985c703-eaf1-

In [13]:
vectorstore.get_by_ids(['5d828406-15a7-4bf7-882a-b92208a010cf'])

[Document(id='5d828406-15a7-4bf7-882a-b92208a010cf', metadata={}, page_content="make the appropriate adjustments. And there are people that will that read David's book and your book, and we'll see the possibility of doing something differently with a terrible childhood or a brutal addiction. And, you know, I think we need more stories of success. I think it's easy to look out there and see all the things that are going wrong and we need to keep paying attention to those, but we need these beacons that draw people forward. And I say that from a place of experience, I mean,")]

### Step 2 : Retrieval

In [14]:
retreiver = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [15]:
retreiver

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x790b127ab3e0>, search_kwargs={'k': 4})

In [20]:
retreiver.invoke('What is this podcast about?')

[Document(id='e653f109-560c-4808-93c7-770b52ac3a1a', metadata={}, page_content="Hey everybody, welcome to the podcast. My guest today is Dr. Andrew Huberman. Andrew is a neuroscientist. He's a Neurobiology Professor at Stanford Medical School and McKnight Foundation and Pew Foundation Fellow and the founder of Huberman Lab, where he's involved in all kinds of really amazing breakthrough research on brain function, brain plasticity and brain regeneration. His work has been published in top journals like Nature. He's been featured in publications like Time, Scientific America"),
 Document(id='21c4acf7-6d18-492e-8d33-64df5d46e088', metadata={}, page_content="for like the last nine months, right? And I just couldn't, couldn't get it together. Like it's a collaborative project. So there's a lot of different people that are involved in this, and they've been working diligently sort of daily, you know putting this thing together. And I've just been focusing on the podcast and been unable to i

### Step 3 : Augmentation

In [21]:
prompt = PromptTemplate(
    template="""Based on the context below, answer the question directly and concisely, And say that "You dont know" if question is ut of context .

Context: {context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)


In [23]:
question = "What is this podcast about?"
retreived_docs = retreiver.invoke(question)

In [24]:
retreived_docs

[Document(id='e653f109-560c-4808-93c7-770b52ac3a1a', metadata={}, page_content="Hey everybody, welcome to the podcast. My guest today is Dr. Andrew Huberman. Andrew is a neuroscientist. He's a Neurobiology Professor at Stanford Medical School and McKnight Foundation and Pew Foundation Fellow and the founder of Huberman Lab, where he's involved in all kinds of really amazing breakthrough research on brain function, brain plasticity and brain regeneration. His work has been published in top journals like Nature. He's been featured in publications like Time, Scientific America"),
 Document(id='21c4acf7-6d18-492e-8d33-64df5d46e088', metadata={}, page_content="for like the last nine months, right? And I just couldn't, couldn't get it together. Like it's a collaborative project. So there's a lot of different people that are involved in this, and they've been working diligently sort of daily, you know putting this thing together. And I've just been focusing on the podcast and been unable to i

In [25]:
context_text = "\n\n".join(doc.page_content for doc in retreived_docs)
context_text

"Hey everybody, welcome to the podcast. My guest today is Dr. Andrew Huberman. Andrew is a neuroscientist. He's a Neurobiology Professor at Stanford Medical School and McKnight Foundation and Pew Foundation Fellow and the founder of Huberman Lab, where he's involved in all kinds of really amazing breakthrough research on brain function, brain plasticity and brain regeneration. His work has been published in top journals like Nature. He's been featured in publications like Time, Scientific America\n\nfor like the last nine months, right? And I just couldn't, couldn't get it together. Like it's a collaborative project. So there's a lot of different people that are involved in this, and they've been working diligently sort of daily, you know putting this thing together. And I've just been focusing on the podcast and been unable to immerse myself in this project because I know from past book projects, when I go in, I go all in, like the addict in me kicks in and it's like, it just becomes 

In [26]:
final_prompt = prompt.format(context=context_text, question=question)
final_prompt

'Based on the context below, answer the question directly and concisely, And say that "You dont know" if question is ut of context .\n\nContext: Hey everybody, welcome to the podcast. My guest today is Dr. Andrew Huberman. Andrew is a neuroscientist. He\'s a Neurobiology Professor at Stanford Medical School and McKnight Foundation and Pew Foundation Fellow and the founder of Huberman Lab, where he\'s involved in all kinds of really amazing breakthrough research on brain function, brain plasticity and brain regeneration. His work has been published in top journals like Nature. He\'s been featured in publications like Time, Scientific America\n\nfor like the last nine months, right? And I just couldn\'t, couldn\'t get it together. Like it\'s a collaborative project. So there\'s a lot of different people that are involved in this, and they\'ve been working diligently sort of daily, you know putting this thing together. And I\'ve just been focusing on the podcast and been unable to immerse

### Step 4: Generation

In [32]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Using distilgpt2 (small, fast)
model_id = "distilgpt2"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.2,
    do_sample=True,
    device=-1  # CPU
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [36]:
answer = llm.invoke(final_prompt)
answer_only = answer.split("Question: What is this podcast about?")[-1].strip()
print("Question: ", question)
print( answer_only)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:  What is this podcast about?
Answer: Well, what does your name mean by 'Brain-Functional' or “Neurological”? Is any one word related to neural activity (neuron)? Do you have anything else with them as well? Or would you prefer something more specific than those words which can only describe neurons within their own area rather than being described elsewhere inside its field itself? You might also want to check out our website www.brainfunctionality.com
